In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pyro
from pyro.infer.autoguide import AutoLowRankMultivariateNormal, init_to_median
from pyro.infer import Predictive
import pyro.distributions as dist
from pyro.optim import Adam
import random
import pandas as pd
import scipy.io as sio
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from scipy.signal import savgol_filter
import gc
import os


# import user-defined files
from utils.Reservoir import Reservoir
from utils.ESNVariational import ESNVariational
from utils.ESNDataset import ESNDataset
from utils.prediction_curve import prediction_curve
from utils.construct_dataset import construct_dataset
from utils.methods_2 import evaluate_metrics
from utils.recursive_inference import recursive_inference
from utils.zoomed_plots import zoomed_plots

## Instantiate the model ESN + SVI

In [ ]:
# PARAMETERS

N_NEURONS = 50
INPUT_DIM = 119 # equals to output dim

LR = 0.001
act_name = nn.Tanh() #if best_params['activation'] == 'Tanh()' else nn.ReLU()
EPOCHS = 100

# Setup Optimizer
optimizer = pyro.optim.Adam({"lr": LR})

In [10]:
## PYRO MODEL
# states is a batch of states of size (batch dimension, neurons per state)
def model_fn(states, scale = 0.1, targets=None):
    N_neurons = states.shape[1]
    # We sample the Readout Weights R
    # .to_event(2) ensures Pyro treats this as a matrix, not independent scalars
    R = pyro.sample("R", dist.Normal(torch.zeros(N_neurons, 119), 1.0).to_event(2))
    
    # Linear projection to find the mean load
    mu = (states @ R)
    
    # Observation noise (how much we expect the real data to fluctuate around mu)
    sigma = pyro.sample("sigma", dist.HalfNormal(scale))
    
    with pyro.plate("data", states.shape[0]):
        return pyro.sample("obs", dist.Normal(mu, sigma).to_event(1), obs=targets)

In [11]:
# PYRO GUIDE

RANK = int(np.sqrt(N_NEURONS))  # Rank for the Multivariate Normal
guide = AutoLowRankMultivariateNormal(model_fn, rank=RANK)

In [ ]:
# Initialize the ESNVariational
esn_var = ESNVariational(
    N_NEURONS, 
    INPUT_DIM, 
    model_fn, 
    guide, 
    optimizer, 
    spectral_radius = 0.8,
    scale = 1.0 # std of the weights we sample
)
esn_var.reservoir.activation = act_name

## Train

In [ ]:
# Load the MAT file for training and validation
mat_file = sio.loadmat('./real_data/100307.REST1.LR.SchaeferS.ptseries.mat')
data = torch.from_numpy(mat_file['tseries']).float()

In [14]:
train_ds, _ = construct_dataset(esn_var.reservoir, data.T, train_split=1.0, burnin=100, noise = False)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

In [15]:
# retrain on the full first dataset
history = esn_var.train(train_loader, EPOCHS)

Training Pyro ESN:   1%|          | 1/100 [00:05<09:25,  5.71s/it, Loss=1090.7594]

Epoch 0 | Total ELBO Loss: 1090.7594


Training Pyro ESN: 100%|██████████| 100/100 [00:46<00:00,  2.15it/s, Loss=628.7537]


## Test --> Inference (y_samples are all the samples drawn for each real point in the inference process, y_mean is the mean of each 1000)

In [ ]:
# fetch test dataset
data_test = torch.from_numpy(sio.loadmat('./real_data/100307.REST2.LR.SchaeferS.ptseries.mat')['tseries']).float()
# Generate Torch datasets
test_ds, _ = construct_dataset(esn_var.reservoir, data_test.T, train_split=1.0, burnin=0, noise = False)

In [ ]:
# predict and plot predictions for different regions
scaler = {"mean": test_ds.mean, "std": test_ds.std} # extract mean for scaling (used for plotting)
y_true, y_samples, y_mean, upper, lower = prediction_curve(esn_var, test_ds, plot = False, scaler = scaler, num_samples = 1000)
#zoomed_plots(0,y_mean,y_true,upper,lower,steps = 200, n_plots=6, rows=3)